<a href="https://colab.research.google.com/github/ajaikumar1905/AJAIKUMAR-S/blob/main/NLP_Text_%26_Audio_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
!pip install tensorflow nltk SpeechRecognition gTTS pydub

In [22]:
import json
import numpy as np
import random
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
import speech_recognition as sr
from gtts import gTTS
from IPython.display import Audio, display

# Download required NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

lemmatizer = WordNetLemmatizer()

# 1. Inbuilt Integrated Default Dataset
intents_json = {
    "intents": [
        {"tag": "greeting", "patterns": ["Hi", "Hello", "Hey there", "Good morning"], "responses": ["Hello! How can I assist you today?", "Hi there! I am listening."]},
        {"tag": "goodbye", "patterns": ["Bye", "See you later", "Goodbye"], "responses": ["Goodbye! Have a great day.", "Talk to you later!"]},
        {"tag": "status", "patterns": ["How are you", "Are you online", "What is your status"], "responses": ["I am fully operational and ready to help.", "All systems are running perfectly!"]}
    ]
}

# 2. Data Preprocessing
words = []
classes = []
documents = []
ignore_words = ['?', '!']

for intent in intents_json['intents']:
    for pattern in intent['patterns']:
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        documents.append((w, intent['tag']))
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

# 3. Create Training Data
training = []
output_empty = [0] * len(classes)

for doc in documents:
    bag = []
    pattern_words = doc[0]
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)

    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1
    training.append([bag, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)
train_x = list(training[:, 0])
train_y = list(training[:, 1])

# 4. Build and Train the Keras Model
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Updated optimizer (decay is deprecated, using learning_rate schedule is better, but this works for basic setup)
sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

print("Training model...")
model.fit(np.array(train_x), np.array(train_y), epochs=150, batch_size=5, verbose=0)
print("Model trained and ready!")

# 5. Functions for prediction and audio
def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

def bow(sentence, words):
    sentence_words = clean_up_sentence(sentence)
    bag = [0]*len(words)
    for s in sentence_words:
        for i,w in enumerate(words):
            if w == s:
                bag[i] = 1
    return(np.array(bag))

def predict_class(sentence, model):
    p = bow(sentence, words)
    res = model.predict(np.array([p]), verbose=0)[0]
    ERROR_THRESHOLD = 0.25
    results = [[i,r] for i,r in enumerate(res) if r>ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)

    return_list = []
    for r in results:
        return_list.append({"intent": classes[r[0]], "probability": str(r[1])})
    return return_list

def get_response(ints, intents_json):
    if not ints:
        return "I'm sorry, I didn't understand that."
    tag = ints[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if(i['tag']== tag):
            result = random.choice(i['responses'])
            break
    return result

# --- Core Audio/Text Processing ---
def process_voice_chat(audio_file_path):
    # 1. Speech to Text
    r = sr.Recognizer()
    with sr.AudioFile(audio_file_path) as source:
        print("Processing audio file...")
        audio_data = r.record(source)
        try:
            user_text = r.recognize_google(audio_data)
            print(f"🗣️ You said: {user_text}")
        except sr.UnknownValueError:
            print("Could not understand the audio.")
            return
        except sr.RequestError:
            print("Could not request results from Google Speech Recognition service.")
            return

    # 2. Text to NLP Model
    ints = predict_class(user_text, model)
    bot_response = get_response(ints, intents_json)
    print(f"🤖 Chatbot: {bot_response}")

    # 3. Text to Speech (Audio Output)
    tts = gTTS(text=bot_response, lang='en')
    audio_output_path = "bot_reply.mp3"
    tts.save(audio_output_path)

    # Display audio player in Colab
    display(Audio(audio_output_path, autoplay=True))

# To test this, you will need to uncomment the line below and provide a valid .wav file
# process_voice_chat('test_audio.wav')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training model...
Model trained and ready!


In [23]:
# Generate a sample audio file with speech for testing
from gtts import gTTS
from pydub import AudioSegment
import os

text_to_speak = "Hello there"
tts = gTTS(text=text_to_speak, lang='en')
tts.save("temp_audio.mp3")

# Convert mp3 to wav
audio = AudioSegment.from_mp3("temp_audio.mp3")
audio.export("sample_audio.wav", format="wav")

os.remove("temp_audio.mp3") # Clean up temp file

print(f"Sample audio file generated as sample_audio.wav with text: '{text_to_speak}'")

Sample audio file generated as sample_audio.wav with text: 'Hello there'


In [24]:
# Test the chatbot with the newly generated speech audio file
process_voice_chat('sample_audio.wav')

Processing audio file...
🗣️ You said: hello there
🤖 Chatbot: Hi there! I am listening.


In [25]:
pip install requests

Replace `requests` with the name of the library you wish to install. You can also install multiple libraries at once by listing them separated by spaces, like this:

```python
!pip install library1 library2 library3
```